# Topic 5: Complex Types — Arrays, Maps, and Structs

> **Phase 3 → Week 4 → Topic 5**

---

## Why Complex Types?

Real-world data is rarely flat. You encounter:
- **JSON APIs**: `{"user": {"id": 1, "tags": ["admin", "vip"]}}` — nested structs + arrays
- **Event logs**: `{"events": [{"type": "click", "ts": 123}, {"type": "buy", "ts": 456}]}`
- **Spark groupBy results**: `collect_list("item")` → ArrayType column
- **Key-value data**: `{"prefs": {"theme": "dark", "lang": "en"}}` → MapType

Spark supports three complex types:

| Type | Python equivalent | Schema type | Example |
|------|------------------|-------------|--------|
| **ArrayType** | list | `ArrayType(IntegerType())` | `[1, 2, 3]` |
| **MapType** | dict | `MapType(StringType(), IntegerType())` | `{"a": 1}` |
| **StructType** | namedtuple/dict | `StructType([StructField(...)])` | `{"id": 1, "name": "x"}` |

---

## Interview Cheat Sheet

**Q: What does explode() do and when would you use it?**
> `explode(array_col)` converts one row containing an array of N elements into N separate rows, one per element. The other columns are duplicated for each row. Use it when you need to operate on individual array elements, or when normalizing nested JSON into a flat table. `posexplode()` also returns the array index.

**Q: What's the difference between explode and explode_outer?**
> `explode()` drops rows where the array is null or empty. `explode_outer()` keeps those rows with null values in the exploded column — like a LEFT OUTER join vs INNER join behavior.

**Q: How do you access a struct field in Spark?**
> Use dot notation: `df.select("address.city")` or `F.col("address.city")`. For nested structs: `"address.geo.lat"`. You can also use `getField("city")` programmatically when the column name is in a variable.

**Q: How do you flatten a deeply nested JSON in Spark?**
> Select individual struct fields using dot notation and give them flat aliases: `F.col("user.address.city").alias("city")`. For arrays, use `explode()` to unnest, then select the struct fields from the exploded column.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("Week4 - Complex Types") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.sql.ansi.enabled", "false") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Ready")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/13 16:01:07 WARN Utils: Your hostname, kali, resolves to a loopback address: 127.0.1.1; using 10.237.41.61 instead (on interface wlan0)
26/06/13 16:01:07 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/06/13 16:01:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Ready


## Part 1: ArrayType

In [2]:
# Create DataFrame with ArrayType column
user_df = spark.createDataFrame([
    (1, "Alice", ["python", "spark", "sql"]),
    (2, "Bob",   ["java", "scala"]),
    (3, "Carol", ["python", "pandas", "spark", "ml"]),
    (4, "Dave",  []),
    (5, "Eve",   None),
], ["id", "name", "skills"])

print("Schema — skills is ArrayType:")
user_df.printSchema()
user_df.show(truncate=False)

Schema — skills is ArrayType:
root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- skills: array (nullable = true)
 |    |-- element: string (containsNull = true)



+---+-----+---------------------------+
|id |name |skills                     |
+---+-----+---------------------------+
|1  |Alice|[python, spark, sql]       |
|2  |Bob  |[java, scala]              |
|3  |Carol|[python, pandas, spark, ml]|
|4  |Dave |[]                         |
|5  |Eve  |NULL                       |
+---+-----+---------------------------+



In [3]:
# Array operations
array_ops = user_df.select(
    "name",
    F.size("skills").alias("skill_count"),
    F.array_contains("skills", "spark").alias("knows_spark"),
    F.element_at("skills", 1).alias("first_skill"),   # 1-indexed!
    F.array_sort("skills").alias("skills_sorted"),
    F.array_distinct(F.concat("skills", F.array(F.lit("python")))).alias("with_python_deduped"),
    F.slice("skills", 1, 2).alias("first_two"),        # slice(col, start, length)
)

print("Array operations:")
array_ops.show(truncate=False)

Array operations:


+-----+-----------+-----------+-----------+---------------------------+---------------------------+----------------+
|name |skill_count|knows_spark|first_skill|skills_sorted              |with_python_deduped        |first_two       |
+-----+-----------+-----------+-----------+---------------------------+---------------------------+----------------+
|Alice|3          |true       |python     |[python, spark, sql]       |[python, spark, sql]       |[python, spark] |
|Bob  |2          |false      |java       |[java, scala]              |[java, scala, python]      |[java, scala]   |
|Carol|4          |true       |python     |[ml, pandas, python, spark]|[python, pandas, spark, ml]|[python, pandas]|
|Dave |0          |false      |NULL       |[]                         |[python]                   |[]              |
|Eve  |-1         |NULL       |NULL       |NULL                       |NULL                       |NULL            |
+-----+-----------+-----------+-----------+---------------------

In [4]:
# explode — one row per array element
print("BEFORE explode:")
user_df.filter(F.col("skills").isNotNull() & (F.size("skills") > 0)) \
       .show(truncate=False)

print("AFTER explode(skills) — one row per skill:")
exploded = user_df.withColumn("skill", F.explode("skills"))
exploded.select("id", "name", "skill").show()

print(f"Original rows: {user_df.count()}, After explode: {exploded.count()} (null/empty dropped)")

BEFORE explode:


+---+-----+---------------------------+
|id |name |skills                     |
+---+-----+---------------------------+
|1  |Alice|[python, spark, sql]       |
|2  |Bob  |[java, scala]              |
|3  |Carol|[python, pandas, spark, ml]|
+---+-----+---------------------------+

AFTER explode(skills) — one row per skill:


+---+-----+------+
| id| name| skill|
+---+-----+------+
|  1|Alice|python|
|  1|Alice| spark|
|  1|Alice|   sql|
|  2|  Bob|  java|
|  2|  Bob| scala|
|  3|Carol|python|
|  3|Carol|pandas|
|  3|Carol| spark|
|  3|Carol|    ml|
+---+-----+------+



Original rows: 5, After explode: 9 (null/empty dropped)


In [5]:
# explode_outer — keeps null/empty array rows
print("explode_outer — keeps rows with empty/null arrays:")
user_df.withColumn("skill", F.explode_outer("skills")) \
       .select("name", "skill") \
       .show()

# posexplode — also returns the array index
print("posexplode — returns index (pos) and value:")
user_df.filter(F.col("name") == "Carol") \
       .select("name", F.posexplode("skills").alias("idx", "skill")) \
       .show()

explode_outer — keeps rows with empty/null arrays:


+-----+------+
| name| skill|
+-----+------+
|Alice|python|
|Alice| spark|
|Alice|   sql|
|  Bob|  java|
|  Bob| scala|
|Carol|python|
|Carol|pandas|
|Carol| spark|
|Carol|    ml|
| Dave|  NULL|
|  Eve|  NULL|
+-----+------+

posexplode — returns index (pos) and value:


+-----+---+------+
| name|idx| skill|
+-----+---+------+
|Carol|  0|python|
|Carol|  1|pandas|
|Carol|  2| spark|
|Carol|  3|    ml|
+-----+---+------+



In [6]:
# collect_list and collect_set — REVERSE of explode (aggregate into array)
# Real-world: group user actions into a session list
orders = spark.read.csv("data/orders.csv", header=True, inferSchema=True)

customer_orders = orders.groupBy("customer_id").agg(
    F.collect_list("order_id").alias("all_order_ids"),
    F.collect_set("status").alias("unique_statuses"),
    F.count("order_id").alias("total_orders")
)

print("collect_list and collect_set:")
customer_orders.show(truncate=False)

collect_list and collect_set:


+-----------+------------------+-----------------------+------------+
|customer_id|all_order_ids     |unique_statuses        |total_orders|
+-----------+------------------+-----------------------+------------+
|6          |[O008, O015]      |[delivered]            |2           |
|9          |[O011]            |[cancelled]            |1           |
|2          |[O003, O017]      |[delivered]            |2           |
|4          |[O006, O020]      |[delivered, shipped]   |2           |
|5          |[O007, O018]      |[delivered]            |2           |
|10         |[O012, O019]      |[processing, delivered]|2           |
|1          |[O001, O002, O013]|[delivered]            |3           |
|3          |[O004, O005, O014]|[delivered, shipped]   |3           |
|7          |[O009]            |[processing]           |1           |
|8          |[O010]            |[delivered]            |1           |
|11         |[O016]            |[delivered]            |1           |
+-----------+-------

## Part 2: MapType

In [7]:
# MapType — key-value pairs in a column (like a Python dict per row)
config_df = spark.createDataFrame([
    (1, "Alice", {"theme": "dark",  "lang": "en", "timezone": "IST"}),
    (2, "Bob",   {"theme": "light", "lang": "zh", "notifications": "off"}),
    (3, "Carol", {"lang": "en",     "timezone": "EST"}),
    (4, "Dave",  None),
], ["id", "name", "prefs"])

print("Schema with MapType:")
config_df.printSchema()
config_df.show(truncate=False)

Schema with MapType:
root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- prefs: map (nullable = true)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)



+---+-----+--------------------------------------------------+
|id |name |prefs                                             |
+---+-----+--------------------------------------------------+
|1  |Alice|{lang -> en, timezone -> IST, theme -> dark}      |
|2  |Bob  |{lang -> zh, theme -> light, notifications -> off}|
|3  |Carol|{lang -> en, timezone -> EST}                     |
|4  |Dave |NULL                                              |
+---+-----+--------------------------------------------------+



In [8]:
# Map operations
map_ops = config_df.select(
    "name",
    F.col("prefs")["theme"].alias("theme"),            # access by key (returns null if missing)
    F.col("prefs")["lang"].alias("lang"),
    F.map_keys("prefs").alias("pref_keys"),
    F.map_values("prefs").alias("pref_values"),
    F.size("prefs").alias("num_prefs"),
    F.map_contains_key("prefs", "theme").alias("has_theme"),
)

print("Map access and operations:")
map_ops.show(truncate=False)

Map access and operations:


+-----+-----+----+----------------------------+----------------+---------+---------+
|name |theme|lang|pref_keys                   |pref_values     |num_prefs|has_theme|
+-----+-----+----+----------------------------+----------------+---------+---------+
|Alice|dark |en  |[lang, timezone, theme]     |[en, IST, dark] |3        |true     |
|Bob  |light|zh  |[lang, theme, notifications]|[zh, light, off]|3        |true     |
|Carol|NULL |en  |[lang, timezone]            |[en, EST]       |2        |false    |
|Dave |NULL |NULL|NULL                        |NULL            |-1       |NULL     |
+-----+-----+----+----------------------------+----------------+---------+---------+



In [9]:
# explode map — one row per key-value pair
print("Exploded map — one row per preference key-value:")
config_df.select("name", F.explode("prefs").alias("pref_key", "pref_val")) \
         .show()

# cleaner way (same result)
config_df.select("name", F.explode("prefs").alias("key", "val")) \
         .show()


Exploded map — one row per preference key-value:


+-----+-------------+--------+
| name|     pref_key|pref_val|
+-----+-------------+--------+
|Alice|         lang|      en|
|Alice|     timezone|     IST|
|Alice|        theme|    dark|
|  Bob|         lang|      zh|
|  Bob|        theme|   light|
|  Bob|notifications|     off|
|Carol|         lang|      en|
|Carol|     timezone|     EST|
+-----+-------------+--------+



+-----+-------------+-----+
| name|          key|  val|
+-----+-------------+-----+
|Alice|         lang|   en|
|Alice|     timezone|  IST|
|Alice|        theme| dark|
|  Bob|         lang|   zh|
|  Bob|        theme|light|
|  Bob|notifications|  off|
|Carol|         lang|   en|
|Carol|     timezone|  EST|
+-----+-------------+-----+



In [10]:
# Create maps from columns using map() or create_map()
orders2 = spark.read.csv("data/orders.csv", header=True, inferSchema=True)

# Build a map column from two columns
orders_with_map = orders2.select(
    "order_id",
    F.create_map(
        F.lit("status"),  F.col("status"),
        F.lit("amount"),  F.col("amount").cast("string")
    ).alias("order_info")
)

print("create_map from columns:")
orders_with_map.show(5, truncate=False)

create_map from columns:


+--------+----------------------------------------+
|order_id|order_info                              |
+--------+----------------------------------------+
|O001    |{status -> delivered, amount -> 1299.99}|
|O002    |{status -> delivered, amount -> 79.98}  |
|O003    |{status -> delivered, amount -> 89.97}  |
|O004    |{status -> delivered, amount -> 1299.99}|
|O005    |{status -> delivered, amount -> 599.98} |
+--------+----------------------------------------+
only showing top 5 rows


## Part 3: StructType (Nested Structs)

In [11]:
# StructType — nested record (like a row inside a row)
# Common when reading JSON with nested objects

nested_schema = StructType([
    StructField("user_id", IntegerType(), False),
    StructField("name",    StringType(),  True),
    StructField("address", StructType([
        StructField("street",  StringType(), True),
        StructField("city",    StringType(), True),
        StructField("country", StringType(), True),
        StructField("geo", StructType([
            StructField("lat", DoubleType(), True),
            StructField("lon", DoubleType(), True),
        ]), True),
    ]), True),
    StructField("tags", ArrayType(StringType()), True),
])

from pyspark.sql import Row
nested_data = [
    Row(user_id=1, name="Alice",
        address=Row(street="12 MG Road", city="Mumbai", country="India",
                    geo=Row(lat=19.08, lon=72.88)),
        tags=["vip", "gold"]),
    Row(user_id=2, name="Bob",
        address=Row(street="5th Ave", city="New York", country="USA",
                    geo=Row(lat=40.71, lon=-74.01)),
        tags=["standard"]),
]

nested_df = spark.createDataFrame(nested_data, nested_schema)
print("Nested struct schema:")
nested_df.printSchema()
nested_df.show(truncate=False)

Nested struct schema:
root
 |-- user_id: integer (nullable = false)
 |-- name: string (nullable = true)
 |-- address: struct (nullable = true)
 |    |-- street: string (nullable = true)
 |    |-- city: string (nullable = true)
 |    |-- country: string (nullable = true)
 |    |-- geo: struct (nullable = true)
 |    |    |-- lat: double (nullable = true)
 |    |    |-- lon: double (nullable = true)
 |-- tags: array (nullable = true)
 |    |-- element: string (containsNull = true)



+-------+-----+-------------------------------------------+-----------+
|user_id|name |address                                    |tags       |
+-------+-----+-------------------------------------------+-----------+
|1      |Alice|{12 MG Road, Mumbai, India, {19.08, 72.88}}|[vip, gold]|
|2      |Bob  |{5th Ave, New York, USA, {40.71, -74.01}}  |[standard] |
+-------+-----+-------------------------------------------+-----------+



In [12]:
# Access nested struct fields with dot notation
print("Access nested fields with dot notation:")
nested_df.select(
    "name",
    "address.city",
    "address.country",
    "address.geo.lat",
    "address.geo.lon",
    F.size("tags").alias("num_tags")
).show()

# Flatten struct into top-level columns
print("Flattened (all struct fields at top level):")
nested_df.select(
    "user_id",
    "name",
    F.col("address.street").alias("street"),
    F.col("address.city").alias("city"),
    F.col("address.country").alias("country"),
    F.col("address.geo.lat").alias("lat"),
    F.col("address.geo.lon").alias("lon"),
    F.explode("tags").alias("tag")
).show()

Access nested fields with dot notation:


+-----+--------+-------+-----+------+--------+
| name|    city|country|  lat|   lon|num_tags|
+-----+--------+-------+-----+------+--------+
|Alice|  Mumbai|  India|19.08| 72.88|       2|
|  Bob|New York|    USA|40.71|-74.01|       1|
+-----+--------+-------+-----+------+--------+

Flattened (all struct fields at top level):


+-------+-----+----------+--------+-------+-----+------+--------+
|user_id| name|    street|    city|country|  lat|   lon|     tag|
+-------+-----+----------+--------+-------+-----+------+--------+
|      1|Alice|12 MG Road|  Mumbai|  India|19.08| 72.88|     vip|
|      1|Alice|12 MG Road|  Mumbai|  India|19.08| 72.88|    gold|
|      2|  Bob|   5th Ave|New York|    USA|40.71|-74.01|standard|
+-------+-----+----------+--------+-------+-----+------+--------+



In [13]:
# Build a struct from individual columns using struct()
flat_df = spark.createDataFrame([
    (1, "Alice", "Mumbai", "India"),
    (2, "Bob",   "NYC",    "USA"),
], ["id", "name", "city", "country"])

# Group city + country into an address struct
with_struct = flat_df.withColumn(
    "address",
    F.struct(F.col("city"), F.col("country"))
).drop("city", "country")

print("Grouping columns into a struct:")
with_struct.printSchema()
with_struct.show(truncate=False)

Grouping columns into a struct:
root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- address: struct (nullable = false)
 |    |-- city: string (nullable = true)
 |    |-- country: string (nullable = true)



+---+-----+---------------+
|id |name |address        |
+---+-----+---------------+
|1  |Alice|{Mumbai, India}|
|2  |Bob  |{NYC, USA}     |
+---+-----+---------------+



## Part 4: Reading Real Nested JSON

In [14]:
# Write and read a nested JSON file
import json

nested_json = [
    {"order_id": "O001", "customer": {"id": 1, "name": "Alice", "country": "India"},
     "items": [{"product": "Laptop", "qty": 1, "price": 1299.99},
               {"product": "Mouse",  "qty": 2, "price": 29.99}]},
    {"order_id": "O002", "customer": {"id": 2, "name": "Bob", "country": "China"},
     "items": [{"product": "Chair",  "qty": 1, "price": 299.99}]},
]

# Write to file
with open("/tmp/nested_orders.json", "w") as f:
    for row in nested_json:
        f.write(json.dumps(row) + "\n")

# Read with Spark — schema inferred
json_df = spark.read.json("/tmp/nested_orders.json")
print("Auto-inferred schema from nested JSON:")
json_df.printSchema()
json_df.show(truncate=False)

Auto-inferred schema from nested JSON:
root
 |-- customer: struct (nullable = true)
 |    |-- country: string (nullable = true)
 |    |-- id: long (nullable = true)
 |    |-- name: string (nullable = true)
 |-- items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- price: double (nullable = true)
 |    |    |-- product: string (nullable = true)
 |    |    |-- qty: long (nullable = true)
 |-- order_id: string (nullable = true)



+-----------------+-----------------------------------------+--------+
|customer         |items                                    |order_id|
+-----------------+-----------------------------------------+--------+
|{India, 1, Alice}|[{1299.99, Laptop, 1}, {29.99, Mouse, 2}]|O001    |
|{China, 2, Bob}  |[{299.99, Chair, 1}]                     |O002    |
+-----------------+-----------------------------------------+--------+



In [15]:
# Flatten the nested JSON — normalize into a flat table
print("Flattening nested JSON into a fact-table style:")
json_df.select(
    "order_id",
    F.col("customer.id").alias("customer_id"),
    F.col("customer.name").alias("customer_name"),
    F.col("customer.country").alias("country"),
    F.explode("items").alias("item")
).select(
    "order_id", "customer_id", "customer_name", "country",
    F.col("item.product").alias("product"),
    F.col("item.qty").alias("qty"),
    F.col("item.price").alias("price"),
    (F.col("item.qty") * F.col("item.price")).alias("line_total")
).show()

Flattening nested JSON into a fact-table style:


+--------+-----------+-------------+-------+-------+---+-------+----------+
|order_id|customer_id|customer_name|country|product|qty|  price|line_total|
+--------+-----------+-------------+-------+-------+---+-------+----------+
|    O001|          1|        Alice|  India| Laptop|  1|1299.99|   1299.99|
|    O001|          1|        Alice|  India|  Mouse|  2|  29.99|     59.98|
|    O002|          2|          Bob|  China|  Chair|  1| 299.99|    299.99|
+--------+-----------+-------------+-------+-------+---+-------+----------+



## Exercises

1. Create a DataFrame with a `scores` column (ArrayType of integers). Calculate the sum, max, and min of each array row using `aggregate()` or `array functions`.
2. From the `user_df` skills data, find all users who know `"spark"` using `array_contains()`.
3. Using `collect_list` on the orders data, build a DataFrame of customer → list of products they've ordered.
4. Flatten the nested JSON from Part 4 and find the order with the highest total `line_total` per customer.
5. Create a map column from orders where the key is `order_id` and value is `status`, then explode it.

In [16]:
# Exercise 2: Find users who know spark
user_df.filter(F.array_contains(F.col("skills"), "spark")) \
       .select("name", "skills") \
       .show(truncate=False)

+-----+---------------------------+
|name |skills                     |
+-----+---------------------------+
|Alice|[python, spark, sql]       |
|Carol|[python, pandas, spark, ml]|
+-----+---------------------------+



In [17]:
# Exercise 3: Customer → list of products ordered
orders_raw = spark.read.csv("data/orders.csv", header=True, inferSchema=True)
products   = spark.read.csv("data/products.csv", header=True, inferSchema=True)

orders_raw.join(products, on="product_id", how="inner") \
          .groupBy("customer_id") \
          .agg(F.collect_set(products["name"]).alias("products_ordered")) \
          .orderBy("customer_id") \
          .show(truncate=False)

+-----------+----------------------------------------------+
|customer_id|products_ordered                              |
+-----------+----------------------------------------------+
|1          |[Noise Headphones, Python Book, Laptop Pro 15]|
|2          |[Desk Lamp, Wireless Mouse]                   |
|3          |[USB-C Hub, Office Chair, Laptop Pro 15]      |
|4          |[USB-C Hub, Wireless Mouse]                   |
|5          |[Spark Book, Noise Headphones]                |
|6          |[Standing Desk, Python Book]                  |
|7          |[Spark Book]                                  |
|8          |[Wireless Mouse]                              |
|9          |[Docker Book]                                 |
|10         |[Office Chair, Laptop Pro 15]                 |
|11         |[Laptop Pro 15]                               |
+-----------+----------------------------------------------+

